In [1]:
import pandas as pd
import networkx as nx

# Ruta directa desde la raíz de sigma-cronomx
ruta_limpia = "backend/gtfs_limpio/"

stops_df = pd.read_csv(ruta_limpia + "stops.csv")
print(f"¡Éxito! Total de paradas cargadas: {len(stops_df)}")

¡Éxito! Total de paradas cargadas: 568


In [2]:
import pandas as pd
import numpy as np

ruta_limpia = "backend/gtfs_limpio/"

# 1. Cargar el resto de los datos de tiempos y rutas
stop_times_df = pd.read_csv(ruta_limpia + "stop_times.csv")
trips_df = pd.read_csv(ruta_limpia + "trips.csv")

# 2. Definir los 3 municipios piloto (Cambia los nombres por los reales de su proyecto)
municipios_piloto = ['Ecatepec', 'Iztapalapa', 'Cuauhtémoc']

# (Nota para Max: En tu modelo final, estos tiempos saldrán del algoritmo de ruta más corta de NetworkX.
# Por ahora, los definimos basados en promedios estimados para probar que la fórmula matemática cuadre).
datos_piloto = pd.DataFrame({
    'municipio': municipios_piloto,
    't_trayecto_min': [75.0, 50.0, 20.0],  # Tiempo en movimiento (arriba del Metro/MB)
    't_espera_min': [25.0, 15.0, 5.0],     # Tiempos muertos de espera (frecuencia)
    't_transbordo_min': [20.0, 10.0, 0.0]  # Tiempo caminando entre estaciones
})

# 3. Aplicar la Fórmula Matemática (Brecha Decompuesta)
# Definimos el presupuesto de tiempo diario que no debería rebasarse (ej. 2 horas)
T_disponible = 120.0

# Calculamos el tiempo total expropiado por el transporte público
tiempo_total_viaje = datos_piloto['t_trayecto_min'] + datos_piloto['t_espera_min'] + datos_piloto['t_transbordo_min']

# Calculamos el índice dividiendo el tiempo gastado entre el tiempo disponible
datos_piloto['IPT_gap'] = np.divide(tiempo_total_viaje, T_disponible)

# 4. Mostrar resultados para el equipo
print("--- Resultados del Índice de Pobreza de Tiempo (IPT_gap) ---")
print(datos_piloto[['municipio', 'IPT_gap']])

# Opcional: Clasificar si están en "Pobreza de Tiempo" (índice > 1.0)
datos_piloto['Pobreza_de_Tiempo'] = datos_piloto['IPT_gap'] > 1.0
print("\n--- Diagnóstico de Pobreza de Tiempo ---")
print(datos_piloto[['municipio', 'Pobreza_de_Tiempo']])

--- Resultados del Índice de Pobreza de Tiempo (IPT_gap) ---
    municipio   IPT_gap
0    Ecatepec  1.000000
1  Iztapalapa  0.625000
2  Cuauhtémoc  0.208333

--- Diagnóstico de Pobreza de Tiempo ---
    municipio  Pobreza_de_Tiempo
0    Ecatepec              False
1  Iztapalapa              False
2  Cuauhtémoc              False


In [3]:
# Variables adicionales para las fórmulas 1 y 3 (Datos representativos para el piloto)
datos_piloto['t_obs'] = tiempo_total_viaje  # Usamos el tiempo total calculado en la celda anterior

# --- Fórmula 1: Índice Relativo de Fricción Temporal (IPT_rel) ---
# t_norm = Tiempo máximo deseable (ej. 45 min)
t_norm = 45.0
alpha = 0.5  # Sensibilidad a la saturación

# Simulamos el factor de saturación (Afluencia_histórica / Capacidad)
# Ecatepec (muy saturado: 1.2), Iztapalapa (al límite: 1.0), Cuauhtémoc (con margen: 0.8)
datos_piloto['A_C'] = [1.2, 1.0, 0.8] 

# Aplicación: IPT_rel = (t_obs / t_norm) * (1 + alpha * (A/C))
datos_piloto['IPT_rel'] = (datos_piloto['t_obs'] / t_norm) * (1 + alpha * datos_piloto['A_C'])


# --- Fórmula 3: Índice Compuesto de Accesibilidad y Vulnerabilidad (IPT_comp) ---
# Simulamos el Índice de Marginación de CONAPO (0 = muy bajo, 1 = muy alto)
datos_piloto['IM'] = [0.7, 0.4, 0.1] 

# Simulamos la "Accesibilidad Relativa" (qué porcentaje de oportunidades alcanzan respecto al mejor municipio)
# Ecatepec (accede al 20%), Iztapalapa (50%), Cuauhtémoc (accede al 100%)
datos_piloto['Acc_Relativa'] = [0.2, 0.5, 1.0]

# Aplicación: IPT_comp = (1 - Acc_Relativa) * (1 + IM)
datos_piloto['IPT_comp'] = (1 - datos_piloto['Acc_Relativa']) * (1 + datos_piloto['IM'])

# --- Mostrar los resultados comparativos ---
print("--- Comparativa de Modelos de Pobreza de Tiempo (Piloto) ---")
print(datos_piloto[['municipio', 'IPT_gap', 'IPT_rel', 'IPT_comp']])

--- Comparativa de Modelos de Pobreza de Tiempo (Piloto) ---
    municipio   IPT_gap   IPT_rel  IPT_comp
0    Ecatepec  1.000000  4.266667      1.36
1  Iztapalapa  0.625000  2.500000      0.70
2  Cuauhtémoc  0.208333  0.777778      0.00


In [4]:
import pandas as pd
import numpy as np

# ==============================================================================
# SCRIPT DE PRUEBA PILOTO: ÍNDICES DE POBREZA DE TIEMPO (IPT)
# Autor: Max
# Equipo: CronoMX / SIGMA (Miguel, David, Gael)
# Objetivo: Sanity check de los 3 candidatos matemáticos antes de conectar NetworkX.
# ==============================================================================

ruta_limpia = "backend/gtfs_limpio/"

# 1. Carga de datos base (Simulación con tiempos fijos para el piloto)
municipios_piloto = ['Ecatepec', 'Iztapalapa', 'Cuauhtémoc']

datos_piloto = pd.DataFrame({
    'municipio': municipios_piloto,
    't_trayecto_min': [75.0, 50.0, 20.0],  # Tiempo en movimiento estimado
    't_espera_min': [25.0, 15.0, 5.0],     # Tiempos muertos de espera
    't_transbordo_min': [20.0, 10.0, 0.0]  # Tiempo caminando entre estaciones
})

# Tiempo total del viaje
datos_piloto['t_obs'] = datos_piloto['t_trayecto_min'] + datos_piloto['t_espera_min'] + datos_piloto['t_transbordo_min']

# ==============================================================================
# CANDIDATO 1: Brecha Decompuesta de Tiempo Libre (IPT_gap)
# ==============================================================================
T_disponible = 120.0  # Presupuesto de movilidad diario en minutos (2 hrs)
datos_piloto['IPT_gap'] = np.divide(datos_piloto['t_obs'], T_disponible)

# Evaluamos si entran en Pobreza de Tiempo (Consume el 100% o más del presupuesto)
datos_piloto['Pobreza_de_Tiempo'] = datos_piloto['IPT_gap'] >= 1.0

# NOTA PARA EL EQUIPO (Interpretación IPT_gap):
# Si el valor es >= 1.0 (como Ecatepec), significa que el transporte público expropia 
# todo el tiempo disponible del usuario, invadiendo sus horas de descanso/ocio.
# Valores bajos (como Cuauhtémoc) indican que el usuario viaja sin sacrificar su calidad de vida.

# ==============================================================================
# CANDIDATO 2: Índice Relativo de Fricción Temporal (IPT_rel)
# ==============================================================================
t_norm = 45.0  # Tiempo máximo deseable de traslado
alpha = 0.5    # Sensibilidad al hacinamiento/saturación

# A_C: Factor de saturación (Afluencia histórica vs Capacidad del sistema)
datos_piloto['A_C'] = [1.2, 1.0, 0.8] 

datos_piloto['IPT_rel'] = (datos_piloto['t_obs'] / t_norm) * (1 + alpha * datos_piloto['A_C'])

# NOTA PARA EL EQUIPO (Interpretación IPT_rel):
# Mide cuánto se "siente" el viaje. Ecatepec arrojará > 4.0, indicando que el traslado 
# castiga al usuario como si fuera 4 veces más largo que el ideal, agravado por 
# viajar en un sistema operando al 120% de capacidad.

# ==============================================================================
# CANDIDATO 3: Índice Compuesto de Accesibilidad y Vulnerabilidad (IPT_comp)
# ==============================================================================
# IM: Índice de Marginación de CONAPO (0 = muy bajo, 1 = muy alto)
datos_piloto['IM'] = [0.7, 0.4, 0.1] 

# Acc_Relativa: % de oportunidades alcanzadas respecto al municipio mejor conectado
datos_piloto['Acc_Relativa'] = [0.2, 0.5, 1.0]

datos_piloto['IPT_comp'] = (1 - datos_piloto['Acc_Relativa']) * (1 + datos_piloto['IM'])

# NOTA PARA EL EQUIPO (Interpretación IPT_comp):
# Este modelo castiga doble a las periferias. Ecatepec sale muy alto porque 
# no solo está lejos de las oportunidades (baja Acc_Relativa), sino que su 
# entorno sociodemográfico es vulnerable (alto IM). 
# Cuauhtémoc da 0.0 porque tiene todo el acceso y baja marginación.

# ==============================================================================
# MOSTRAR RESULTADOS GLOBALES
# ==============================================================================
print("--- Resultados del Sanity Check: Municipios Piloto ---")
print(datos_piloto[['municipio', 'IPT_gap', 'Pobreza_de_Tiempo', 'IPT_rel', 'IPT_comp']])

--- Resultados del Sanity Check: Municipios Piloto ---
    municipio   IPT_gap  Pobreza_de_Tiempo   IPT_rel  IPT_comp
0    Ecatepec  1.000000               True  4.266667      1.36
1  Iztapalapa  0.625000              False  2.500000      0.70
2  Cuauhtémoc  0.208333              False  0.777778      0.00
